In [1]:
! pip install ipywidgets nbformat


In [2]:
! pip install -q groq

In [3]:
from groq import Groq


client = Groq(api_key='')

In [4]:
#! pip install langchain
#! pip install langchain_groq
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

chat = ChatGroq(temperature=0, groq_api_key='', model_name="llama3-70b-8192")

## Connecting to neo4j + Query Generation

In [5]:
! pip install neo4j
! pip install langchain
! pip install langchain_community

In [6]:
import tkinter as tk
from tkinter import scrolledtext
from langchain_community.graphs import Neo4jGraph
from langchain.chains import GraphCypherQAChain
from langchain.prompts import PromptTemplate
from langchain.chains.conversation.memory import ConversationBufferMemory

graph = Neo4jGraph(
    url="bolt://localhost:7687",
    username="neo4j",
    password="adminadmin"
)

CYPHER_GENERATION_TEMPLATE = """
You are an expert Neo4j Developer translating user questions into Cypher to answer questions about data provenance.
Convert the user's question based on the schema.

Instructions:
Use only the provided relationship types and properties in the schema.
Do not use any other relationship types or properties that are not provided.

If no data is returned, do not attempt to answer the question.
Only respond to questions that require you to construct a Cypher statement.
Do not include any explanations or apologies in your responses.

Examples:
#Find all the entities
MATCH (n:Entity) RETURN n

#Find all the activities
MATCH (n:Activity) RETURN n

#How many entities were generated by the first activity?
MATCH (first:Activity)
WHERE NOT ()-[:NEXT]->(first)
MATCH (e:Entity)-[:WAS_GENERATED_BY]->(first)
RETURN count(e) AS generatedEntitiesFirstActivity


#Count all communities using louvain on WAS_DERIVED_FROM
CALL gds.graph.drop('proj', false)
YIELD graphName AS droppedGraph
WITH droppedGraph
CALL gds.graph.project('proj', ['Activity'], {{ WAS_DERIVED_FROM:{{orientation:'NATURAL'}} }})
YIELD graphName AS projectedGraph
WITH projectedGraph
CALL gds.louvain.mutate('proj', {{mutateProperty:'communityId'}})
YIELD communityCount
RETURN communityCount AS result

Schema: {schema}
Question: {question}
"""

cypher_generation_prompt = PromptTemplate(
    template=CYPHER_GENERATION_TEMPLATE,
    input_variables=["schema", "question"]
)

cypher_chain = GraphCypherQAChain.from_llm(
    llm=chat,
    graph=graph,
    cypher_prompt=cypher_generation_prompt,
    verbose=True
)

def ask_question():
    question = question_entry.get()
    if question.lower() == "exit":
        root.destroy()
        return

    response = cypher_chain.invoke({"query": question})
    result_text.insert(tk.END, f"\nQ: {question}\nA: {response['result']}\n\n")
    result_text.yview(tk.END)
    question_entry.delete(0, tk.END)

# Creazione dell'interfaccia grafica
root = tk.Tk()
root.title("Neo4j Chat")
root.geometry("700x500")
root.configure(bg="#2c3e50")

frame = tk.Frame(root, bg="#34495e", padx=10, pady=10)
frame.pack(pady=20, padx=20, fill=tk.BOTH, expand=True)

question_entry = tk.Entry(frame, width=70, font=("Arial", 12))
question_entry.pack(pady=10, padx=10)

ask_button = tk.Button(frame, text="Ask", command=ask_question, font=("Arial", 12), bg="#1abc9c", fg="white", relief=tk.FLAT)
ask_button.pack(pady=5)

result_text = scrolledtext.ScrolledText(frame, width=80, height=20, font=("Arial", 12), bg="#ecf0f1", fg="#2c3e50")
result_text.pack(pady=10, padx=10, fill=tk.BOTH, expand=True)

root.mainloop()





> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (e:Entity {id: "entity:07eb17aa-b874-4ebb-94d8-d9bacee8a75e"})-[:WAS_DERIVED_FROM]->(derived:Entity) RETURN derived.value AS derivedEntityValue
Full Context:
[{'derivedEntityValue': nan}]

> Finished chain.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (e:Entity {id: "entity:a9059de5-ca71-494e-9ecb-bfc9a7d76594"})-[:WAS_DERIVED_FROM]->(derived:Entity) RETURN derived.value AS derivedValue
Full Context:
[]

> Finished chain.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (e:Entity {id: "entity:07eb17aa-b874-4ebb-94d8-d9bacee8a75e"})-[:WAS_DERIVED_FROM]->(derived:Entity) RETURN derived.value AS derivedValue
Full Context:
[{'derivedValue': nan}]

> Finished chain.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c1:Column {id: "column:34cffa40-c3c9-4484-ad96-13d2712d7ba4"})-[:WAS_DERIVED_FROM]->(c2:Column) RETURN c2.value AS derivedColumnValue
Full Context: